# 03 Transformer Embeddings (Colab GPU)

**Goal:** Generate sentence embeddings for all financial headlines using `all-MiniLM-L6-v2`, aggregate by day using mean pooling, and export as a CSV for use in the local VS Code project.

**Runtime:** Google CoLab T4 GPU

**Data cutoff:** Headlines are filtered to ≤ 2023-12-31 to match the project-wide DATA_END.

## 1. Environment Setup & Data Loading

Install `sentence-transformers`, mount Google Drive, and verify GPU availability. We need a T4 GPU (free Colab tier) to encode ~18k headlines in reasonable time (~1–3 minutes). The raw headlines CSV is loaded from Drive and filtered to our project's DATA_END (2023-12-31).

In [1]:
# Install sentence-transformers
!pip install -q sentence-transformers

In [2]:
import torch
import pandas as pd
import numpy as np
from google.colab import files


# use google colab for GPU access
from google.colab import drive
drive.mount('/content/drive')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Mounted at /content/drive
Using device: cuda


In [3]:
headlines_df = pd.read_csv('/content/drive/MyDrive/FSA_ML_bootcamp/capstone_project/sp500_headlines.csv')

headlines_df.columns = [c.lower() for c in headlines_df.columns]
headlines_df['date'] = pd.to_datetime(headlines_df['date'])

print(f"Raw headlines loaded: {headlines_df.shape}")
print(f"Date range: {headlines_df['date'].min().date()} → {headlines_df['date'].max().date()}")
print(f"Columns: {headlines_df.columns.tolist()}")
display(headlines_df.head())

Raw headlines loaded: (19127, 3)
Date range: 2008-01-02 → 2024-03-04
Columns: ['title', 'date', 'cp']


,title,date,cp
0,"JPMorgan Predicts 2008 Will Be ""Nothing But Net""",2008-01-02,1447.16
1,Dow Tallies Biggest First-session-of-year Poin...,2008-01-02,1447.16
2,2008 predictions for the S&P 500,2008-01-02,1447.16
3,"U.S. Stocks Higher After Economic Data, Monsan...",2008-01-03,1447.16
4,U.S. Stocks Climb As Hopes Increase For More F...,2008-01-07,1416.18


## 2. Filter to Data Cutoff (2023-12-31)
Remove all headlines after our project-wide DATA_END to stay consistent with SPY/VIX data.

In [4]:
DATA_END = "2023-12-31"

headlines_df = headlines_df[headlines_df['date'] <= DATA_END].copy()

print(f"After cutoff: {headlines_df.shape}")
print(f"Date range: {headlines_df['date'].min().date()} → {headlines_df['date'].max().date()}")

After cutoff: (18294, 3)
Date range: 2008-01-02 → 2023-12-29


## 3. Clean & Deduplicate Headlines
Remove junk (very short strings) and exact duplicates within the same day.

In [5]:
before = len(headlines_df)

# Clean: convert to string, strip whitespace, remove very short headlines
headlines_df['title'] = headlines_df['title'].astype(str).str.strip()
headlines_df = headlines_df[headlines_df['title'].str.len() > 5]

# Remove duplicates: same headline on the same day
# Same headline appearing on different dates is kept
headlines_df = headlines_df.drop_duplicates(subset=['date', 'title'])

after = len(headlines_df)
print(f"Before cleaning: {before}")
print(f"After cleaning:  {after} (removed {before - after})")

Before cleaning: 18294
After cleaning:  17326 (removed 968)


## 4. Load Embedding Model
Using `sentence-transformers/all-MiniLM-L6-v2` — a lightweight 384-dimensional model that runs well on Colab T4 GPU.

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Verify GPU is being used
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Device: cuda
Embedding dimension: 384


## 5. Generate Embeddings (Batch)
Encode all headlines in one batch. This is the GPU-heavy step, should take ~1-3 minutes on T4 for ~18k headlines.

In [7]:
%%time

all_embeddings = model.encode(
    headlines_df['title'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,  # L2 normalize
    device=device,
)

print(f"Embedding matrix shape: {all_embeddings.shape}")

Batches:   0%|          | 0/68 [00:00<?, ?it/s]

Embedding matrix shape: (17326, 384)
CPU times: user 5.79 s, sys: 218 ms, total: 6.01 s
Wall time: 6.15 s


## 6. Aggregate by Day (Mean Pooling)
For each trading day, compute the element-wise mean of all headline embeddings. Also record the headline count and the L2 norm of the mean vector (embedding magnitude).

### How daily Embeddings Work — Example

One day has 3 headlines → each becomes a vector with 384 numbers:

|            | dim_0 | dim_1 | dim_2 | ... | dim_383 |
|------------|-------|-------|-------|-----|---------|
| Headline 1 |  0.1  |  0.2  |  0.2  | ... |   0.08  |
| Headline 2 |  0.5  |  0.7  |  0.1  | ... |  -0.03  |
| Headline 3 |  0.3  |  0.3  |  0.9  | ... |   0.11  |
| **Mean**   |**0.3**|**0.4**|**0.4**| ... | **0.05**|

Mean pooling averages each column → produces **one vector with 384 numbers**.

"Expand Embeddings into Columns" separates this single array into individual columns so it can be saved as a CSV:

| date       | headline_count | emb_0 | emb_1 | emb_2 | ... | emb_383 |
|------------|----------------|-------|-------|-------|-----|---------|
| 2008-01-02 | 3              |  0.3  |  0.4  |  0.4  | ... |   0.05  |

In [8]:
# Attach embeddings to the DataFrame
headlines_df['embedding'] = list(all_embeddings)

# Aggregate by date
def aggregate_daily(group):
    emb_matrix = np.stack(group['embedding'].values)
    mean_emb = emb_matrix.mean(axis=0)
    return pd.Series({
        'mean_embedding': mean_emb, # news direction
        'headline_count': len(group), # how many new on that day
        'embedding_magnitude': np.linalg.norm(mean_emb), # how strong is the direction
    })

daily_news = headlines_df.groupby('date').apply(aggregate_daily).reset_index()

print(f"Unique days with headlines: {len(daily_news)}")
print(f"Date range: {daily_news['date'].min().date()} → {daily_news['date'].max().date()}")
print(f"Headlines per day — mean: {daily_news['headline_count'].mean():.1f}, "
      f"median: {daily_news['headline_count'].median():.0f}, "
      f"max: {daily_news['headline_count'].max()}")

Unique days with headlines: 3464
Date range: 2008-01-02 → 2023-12-29
Headlines per day — mean: 5.0, median: 3, max: 55


/tmp/ipykernel_4265/557918390.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_news = headlines_df.groupby('date').apply(aggregate_daily).reset_index()


## 7. Expand Embeddings into Columns
Convert the mean embedding arrays into 384 separate columns (`emb_0` through `emb_383`) for CSV export.

In [9]:
emb_df = pd.DataFrame(
    daily_news['mean_embedding'].tolist(),
    columns=[f'emb_{i}' for i in range(384)],
    index=daily_news.index,
)

daily_news_features = pd.concat([
    daily_news[['date', 'headline_count', 'embedding_magnitude']],
    emb_df,
], axis=1)

print(f"Final shape: {daily_news_features.shape}")
print(f"Columns: date, headline_count, embedding_magnitude, emb_0 ... emb_383")

Final shape: (3464, 387)
Columns: date, headline_count, embedding_magnitude, emb_0 ... emb_383


## 8. Quick Sanity Checks
Verify no NaN values and spot-check a few embeddings.

In [10]:
# Check for NaN
nan_count = daily_news_features.isna().sum().sum()
print(f"Total NaN values: {nan_count}")

# Spot-check: first 3 days
display(daily_news_features[['date', 'headline_count', 'embedding_magnitude', 'emb_0', 'emb_1', 'emb_2']].head(3))

# Verify embeddings are normalized (L2 norm of each emb row should be close to 1)
sample_norms = np.linalg.norm(daily_news_features[[f'emb_{i}' for i in range(384)]].values[:5], axis=1)
print(f"\nL2 norms of first 5 daily mean embeddings: {sample_norms.round(4)}")
print("(These are means of normalized vectors, so norms will be < 1 — that's expected)")

Total NaN values: 0


,date,headline_count,embedding_magnitude,emb_0,emb_1,emb_2
0,2008-01-02,3,0.730037,-0.017392,-0.034300,0.062104
1,2008-01-03,1,1.000000,0.054529,-0.060522,0.024491
2,2008-01-07,1,1.000000,-0.012102,-0.168724,0.000768



L2 norms of first 5 daily mean embeddings: [0.73   1.     1.     0.7199 1.    ]
(These are means of normalized vectors, so norms will be < 1 — that's expected)


## 9. Save & Download
Save as CSV and download to your local machine. Place the file in `data/processed/daily_news_embeddings.csv`.

---
## 10. Summary

| Output | Value |
|--------|-------|
| **Saved file** | `data/processed/daily_news_embeddings.csv` |
| **Matrix shape** | 3,464 rows × 387 cols (date + headline_count + embedding_magnitude + emb_0…emb_383) |
| **Date range** | 2008-01-02 → 2023-12-29 |
| **Headlines encoded** | 17,326 (after dedup and cutoff filter from 19,127 raw) |
| **Embedding model** | `sentence-transformers/all-MiniLM-L6-v2` — 384-dim, L2-normalized |
| **Aggregation** | Mean pooling per trading day |
| **NaN values** | 0 — clean output |

**Trading day coverage:** 3,464 of ~4,174 trading days have at least one headline (~83%). The remaining ~710 days will receive zero-vector imputation in notebook 04.

**Next step:** Download `daily_news_embeddings.csv` and place it in `data/processed/`. Then run notebook 04 locally to merge with technical features, apply PCA, and build the final feature matrix.

In [11]:
output_filename = 'daily_news_embeddings.csv'
daily_news_features.to_csv(output_filename, index=False)

print(f"Saved: {output_filename}")
print(f"Size: {daily_news_features.shape[0]} rows × {daily_news_features.shape[1]} cols")

# Download to your local machine
files.download(output_filename)

Saved: daily_news_embeddings.csv
Size: 3464 rows × 387 cols


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>